# Phase B Step 6c Only: Residual Computation (Skip Demucs)

**Purpose:** Fast residual computation for 48 mixes that have Demucs stems but no residuals.

This notebook **skips Demucs separation** (06a/06b) entirely and downloads pre-computed stems from HF, then runs only residual computation (06c).

**Why:** 
- All 48 incomplete mixes have stems already on HF (5,309 track stems + 4,864 mix segment stems)
- Only residual computation failed
- Estimated runtime: ~1-2 hours on T4 x2 (vs 2-3 days for full Demucs retry)

**GPU:** T4 x2 recommended (better than P100 for this task — more RAM, good for parallel downloads)

**Manifest split:** Use MANIFEST config in Cell 2 to target "part1", "part2", or "both"

In [ ]:
!pip install -q huggingface_hub soundfile librosa cvxpy torchaudio ecos clarabel

import subprocess, sys, os
# Enable PyTorch memory fragmentation mitigation (fixes GPU OOM on large segments)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

result = subprocess.run(
    ['git', 'clone', '--depth', '1', 'https://github.com/Uday-461/ai-dj-v4.git', '/kaggle/working/ai-dj/v4'],
    capture_output=True, text=True
)
if result.returncode != 0 and 'already exists' not in result.stderr:
    print('Clone failed:', result.stderr)
else:
    print('Repo ready at /kaggle/working/ai-dj/v4')

!pip install -q -e /kaggle/working/ai-dj/v4
sys.path.insert(0, '/kaggle/working/ai-dj/v4')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'PYTORCH_CUDA_ALLOC_CONF: {os.environ.get("PYTORCH_CUDA_ALLOC_CONF", "not set")}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
import os

HF_TOKEN    = "hf_..."                   # <-- paste your HuggingFace token here
HF_REPO     = "Uday-4/djmix-v3"
DATA_ROOT   = "/kaggle/working/djmix"
MANIFEST    = "both"                     # "part1", "part2", or "both"

# -------------------------------------------------------
os.makedirs(DATA_ROOT, exist_ok=True)
os.environ["AIDJ_DATA_ROOT"] = DATA_ROOT

if HF_TOKEN.startswith("hf_..."):
    raise ValueError("Edit Cell 2: set HF_TOKEN to your real HuggingFace token")

from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
try:
    api.whoami()
    print(f'HF auth OK — repo: {HF_REPO}')
except Exception as e:
    raise RuntimeError(f'HF auth failed: {e}')

In [ ]:
import json
import pickle
import shutil
import tempfile
import time
import logging
from pathlib import Path
from collections import defaultdict
from huggingface_hub import hf_hub_download, list_repo_files

# --- Setup logging ---
logging.basicConfig(
    level=logging.INFO,
    format='[%(asctime)s] %(levelname)s: %(message)s',
    datefmt='%H:%M:%S'
)
log = logging.getLogger('06c-only')

DATA_ROOT_PATH = Path(DATA_ROOT)

# --- Load manifests to get target mix IDs ---
manifest_part1_path = "/kaggle/working/ai-dj/v4/data/manifest_100mix_part1.json"
manifest_part2_path = "/kaggle/working/ai-dj/v4/data/manifest_100mix_part2.json"

with open(manifest_part1_path) as f:
    manifest_part1_mixes = {m["id"] for m in json.load(f)["mixes"]}
with open(manifest_part2_path) as f:
    manifest_part2_mixes = {m["id"] for m in json.load(f)["mixes"]}

if MANIFEST == "part1":
    target_mixes = manifest_part1_mixes
    log.info(f'Targeting manifest part1: {len(target_mixes)} mixes')
elif MANIFEST == "part2":
    target_mixes = manifest_part2_mixes
    log.info(f'Targeting manifest part2: {len(target_mixes)} mixes')
else:  # "both"
    target_mixes = manifest_part1_mixes | manifest_part2_mixes
    log.info(f'Targeting both manifests: {len(target_mixes)} mixes')

# --- List everything on HF ---
log.info('Listing HF files...')
all_hf_files = list(list_repo_files(repo_id=HF_REPO, repo_type="dataset", token=HF_TOKEN))
log.info(f'Total files on HF: {len(all_hf_files)}')

# Segment stem dirs on HF: stems/mix_segments/{tran_id}/{stem}.ogg
seg_stem_files = [f for f in all_hf_files if f.startswith("stems/mix_segments/") and f.endswith(".ogg")]
seg_stems_by_tran = defaultdict(set)
for f in seg_stem_files:
    parts = f.split("/")
    tran_id = parts[2]
    stem_name = parts[3].replace(".ogg", "")
    seg_stems_by_tran[tran_id].add(stem_name)

EXPECTED_STEMS = {"drums", "bass", "vocals", "other"}
complete_segments = {tid for tid, stems in seg_stems_by_tran.items() if stems >= EXPECTED_STEMS}

# Residual files on HF: results/residuals/{mix_id}/{tran_id}.npz
residual_files = [f for f in all_hf_files if f.startswith("results/residuals/") and f.endswith(".npz")]
residual_tran_ids = set(f.split("/")[3].replace(".npz", "") for f in residual_files)

# Count residuals PER MIX to identify incomplete mixes
mixes_with_any_residuals = defaultdict(int)
for f in residual_files:
    mix_id = f.split("/")[2]
    mixes_with_any_residuals[mix_id] += 1

# Transition PKLs on HF
transition_pkls = [f for f in all_hf_files if f.startswith("results/transitions/") and f.endswith(".pkl")]

# Load both manifests to get full mix info
manifest_mixes = []
for mp in [manifest_part1_path, manifest_part2_path]:
    with open(mp) as f:
        manifest_mixes.extend(json.load(f)["mixes"])

# Download transition PKLs and find expected segment IDs
log.info('Downloading transition PKLs...')
trans_dir = DATA_ROOT_PATH / "results" / "transitions"
trans_dir.mkdir(parents=True, exist_ok=True)

expected_segments = {}  # tran_id -> {mix_id, tran dict}
mixes_with_transitions = {}  # mix_id -> [transitions]
incomplete_mix_details = {}  # mix_id -> {expected: N, complete: N, residuals: N, transitions: []}

for m in manifest_mixes:
    mix_id = m["id"]
    
    # Filter by manifest
    if mix_id not in target_mixes:
        continue
    
    pkl_path = trans_dir / f"{mix_id}.pkl"
    if not pkl_path.exists():
        try:
            hf_hub_download(
                repo_id=HF_REPO, filename=f"results/transitions/{mix_id}.pkl",
                repo_type="dataset", token=HF_TOKEN,
                local_dir=DATA_ROOT,
            )
        except Exception:
            continue
    if not pkl_path.exists():
        continue
    with open(pkl_path, 'rb') as f:
        transitions = pickle.load(f)
    if not transitions:
        continue
    mixes_with_transitions[mix_id] = transitions
    
    # Track info for incomplete mixes
    if mixes_with_any_residuals.get(mix_id, 0) == 0:
        incomplete_mix_details[mix_id] = {
            'expected': 0,
            'complete': 0,
            'residuals': 0,
            'transitions': []
        }
    
    for t in transitions:
        tran_id = t["tran_id"]
        # Skip segments that would be too short (same logic as step_06b)
        mix_cue_in = t.get("mix_cue_in_time_next")
        mix_cue_out = t.get("mix_cue_out_time_prev")
        if mix_cue_in is None or mix_cue_out is None:
            continue
        from aidj import config
        seg_len = int((max(mix_cue_in, mix_cue_out) - min(mix_cue_in, mix_cue_out)) * config.SR)
        if seg_len < config.SR:
            continue
        expected_segments[tran_id] = {"mix_id": mix_id, "tran": t}
        
        # Track for incomplete mixes
        if mix_id in incomplete_mix_details:
            incomplete_mix_details[mix_id]['expected'] += 1
            incomplete_mix_details[mix_id]['transitions'].append(tran_id)

# Count complete and residual segments per incomplete mix
for mix_id in incomplete_mix_details:
    trans_ids = incomplete_mix_details[mix_id]['transitions']
    incomplete_mix_details[mix_id]['complete'] = len([t for t in trans_ids if t in complete_segments])
    incomplete_mix_details[mix_id]['residuals'] = len([t for t in trans_ids if t in residual_tran_ids])

# CRITICAL: Only target mixes that have NO residuals yet (the 48 incomplete ones)
# If a mix has even 1 residual, it was already processed in original Step 6 — skip it
mixes_needing_residuals = defaultdict(list)
for tran_id, info in expected_segments.items():
    mix_id = info["mix_id"]
    # Only process mixes that have ZERO residuals (truly incomplete, not partially done)
    if mixes_with_any_residuals.get(mix_id, 0) == 0:
        if tran_id in complete_segments and tran_id not in residual_tran_ids:
            mixes_needing_residuals[mix_id].append(info["tran"])

incomplete_mixes_count = len([m for m in target_mixes if mixes_with_any_residuals.get(m, 0) == 0])
complete_mixes_count = len([m for m in target_mixes if mixes_with_any_residuals.get(m, 0) > 0])

log.info('='*70)
log.info('06c-ONLY ANALYSIS')
log.info('='*70)
log.info(f'Total expected segments: {len(expected_segments)}')
log.info(f'Complete stems on HF: {len(complete_segments & set(expected_segments.keys()))}')
log.info(f'Mixes with 0 residuals (incomplete): {incomplete_mixes_count}')
log.info(f'Mixes with ≥1 residuals (already done): {complete_mixes_count}')
log.info(f'Complete stems needing residuals: {sum(len(v) for v in mixes_needing_residuals.values())}')
log.info(f'Mixes to process: {len(mixes_needing_residuals)}')
log.info('='*70)

# DIAGNOSTIC: Show status of all 48 incomplete mixes
log.info('\nDIAGNOSTIC: All 48 incomplete mixes:')
for i, mix_id in enumerate(sorted(incomplete_mix_details.keys()), 1):
    details = incomplete_mix_details[mix_id]
    exp = details['expected']
    comp = details['complete']
    res = details['residuals']
    to_do = sum(1 for t in details['transitions'] if t in complete_segments and t not in residual_tran_ids)
    status = '✓ DONE' if to_do == 0 else '◌ NEEDS WORK'
    log.info(f'  [{i:2d}] {mix_id}: expected={exp} complete={comp} residuals={res} to_do={to_do} {status}')

if mixes_needing_residuals:
    log.info('\n' + '='*70)
    log.info('MIXES TO PROCESS:')
    log.info('='*70)
    for i, mix_id in enumerate(sorted(mixes_needing_residuals), 1):
        trans = mixes_needing_residuals[mix_id]
        log.info(f'  [{i:2d}] {mix_id}: {len(trans)} transitions')
else:
    log.warning('No work to do! All incomplete mixes already have residuals.')

In [ ]:
# Cell 4 — Main: Download stems + compute residuals + upload
# GPU OPTIMIZED for T4 x2

import sys
if '/kaggle/working/ai-dj/v4' not in sys.path:
    sys.path.insert(0, '/kaggle/working/ai-dj/v4')

import logging
import numpy as np
import librosa
from huggingface_hub import hf_hub_download

# Setup logger (use same one from Cell 3)
log = logging.getLogger('06c-only')

from aidj.stems.stem_cache import StemCache
from aidj.data.residual import compute_residual, align_track_to_mix_segment
from aidj import config

if not any(mixes_needing_residuals.values()):
    log.warning('Nothing to process!')
else:
    stem_cache = StemCache(DATA_ROOT_PATH)
    log.info('StemCache initialized')

    def download_track_stems(tid):
        """Download all 4 stem OGGs for a track from HF."""
        stem_dir = DATA_ROOT_PATH / "stems" / "tracks" / tid
        ext = config.STEM_EXT[config.STEM_FORMAT]
        for stem in config.STEMS:
            local = stem_dir / f"{stem}{ext}"
            if local.exists():
                continue
            stem_dir.mkdir(parents=True, exist_ok=True)
            try:
                hf_hub_download(
                    repo_id=HF_REPO,
                    filename=f"stems/tracks/{tid}/{stem}{ext}",
                    repo_type="dataset", token=HF_TOKEN,
                    local_dir=DATA_ROOT,
                )
            except Exception as e:
                log.warning(f'Could not download stem {stem} for track {tid}: {e}')

    session_start = time.time()
    total_residuals = 0
    total_skipped = 0

    all_affected_mixes = sorted(mixes_needing_residuals.keys())
    log.info(f'\nStarting 06c processing for {len(all_affected_mixes)} mixes\n')

    for mix_idx, mix_id in enumerate(all_affected_mixes):
        residuals_to_compute = mixes_needing_residuals[mix_id]
        
        log.info(f'[{mix_idx+1:2d}/{len(all_affected_mixes):2d}] {mix_id}: {len(residuals_to_compute)} residuals')

        transitions = mixes_with_transitions[mix_id]
        res_dir = DATA_ROOT_PATH / "results" / "residuals" / mix_id
        res_dir.mkdir(parents=True, exist_ok=True)

        # Download track stems needed for residual computation
        track_ids = set()
        for t in residuals_to_compute:
            if t.get("track_id_prev"):
                track_ids.add(t["track_id_prev"])
            if t.get("track_id_next"):
                track_ids.add(t["track_id_next"])
        log.info(f'  ↓ Downloading {len(track_ids)} track stems...')
        for tid in sorted(track_ids):
            download_track_stems(tid)

        # Compute residuals
        log.info(f'  ◌ Computing {len(residuals_to_compute)} residuals...')
        res_ok = 0
        for i, tran in enumerate(residuals_to_compute):
            tran_id = tran["tran_id"]
            prev_id = tran["track_id_prev"]
            next_id = tran["track_id_next"]
            out_path = res_dir / f"{tran_id}.npz"

            # Check if already exists (paranoid check)
            if out_path.exists():
                res_ok += 1
                continue

            mix_cue_in = tran.get("mix_cue_in_time_next")
            mix_cue_out = tran.get("mix_cue_out_time_prev")
            if mix_cue_in is None or mix_cue_out is None:
                total_skipped += 1
                continue
            region_len = int((max(mix_cue_in, mix_cue_out) - min(mix_cue_in, mix_cue_out)) * config.SR)
            if region_len < config.SR:
                total_skipped += 1
                continue

            try:
                mix_seg_stems = stem_cache.load_stems("mix_segments", tran_id)
                prev_track_stems = stem_cache.load_stems("tracks", prev_id)
                next_track_stems = stem_cache.load_stems("tracks", next_id)

                if mix_seg_stems is None:
                    log.debug(f'    {tran_id}: missing mix segment stems')
                    total_skipped += 1
                    continue

                residual_data = {}
                if prev_track_stems is not None:
                    t_start = tran.get("track_cue_in_time_prev", 0)
                    aligned = {s: align_track_to_mix_segment(
                        prev_track_stems[s], t_start, region_len, config.SR
                    ) for s in config.STEMS if s in prev_track_stems}
                    for stem, spec in compute_residual(mix_seg_stems, aligned).items():
                        residual_data[f"{stem}_prev"] = spec

                if next_track_stems is not None:
                    t_start = tran.get("track_cue_in_time_next", 0)
                    aligned = {s: align_track_to_mix_segment(
                        next_track_stems[s], t_start, region_len, config.SR
                    ) for s in config.STEMS if s in next_track_stems}
                    for stem, spec in compute_residual(mix_seg_stems, aligned).items():
                        residual_data[f"{stem}_next"] = spec

                if residual_data:
                    np.savez_compressed(str(out_path), **residual_data)
                    res_ok += 1
            except Exception as e:
                log.warning(f'    {tran_id}: {e}')
                total_skipped += 1

        total_residuals += res_ok
        log.info(f'  ✓ Computed {res_ok}/{len(residuals_to_compute)} residuals')

        # --- Upload residuals for this mix ---
        with tempfile.TemporaryDirectory() as staging_dir:
            staging = Path(staging_dir)
            n_files = 0

            if res_dir.exists():
                for f in res_dir.iterdir():
                    dst = staging / f"results/residuals/{mix_id}/{f.name}"
                    dst.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(f, dst)
                    n_files += 1

            if n_files > 0:
                log.info(f'  ⬆ Uploading {n_files} residuals to HF...')
                # Retry logic for HF upload
                max_retries = 3
                for attempt in range(1, max_retries + 1):
                    try:
                        api.upload_large_folder(
                            folder_path=str(staging),
                            repo_id=HF_REPO,
                            repo_type="dataset",
                        )
                        log.info(f'  ✓ Upload complete')
                        break
                    except Exception as e:
                        if attempt < max_retries:
                            log.warning(f'    Upload attempt {attempt} failed, retrying...')
                            time.sleep(5)
                        else:
                            log.error(f'    Upload failed after {max_retries} attempts: {e}')

        # Cleanup local stems to free disk
        for tid in track_ids:
            stem_dir = DATA_ROOT_PATH / "stems" / "tracks" / tid
            if stem_dir.exists():
                shutil.rmtree(stem_dir)

    elapsed = (time.time() - session_start) / 60
    log.info('\n' + '='*70)
    log.info('COMPLETE')
    log.info('='*70)
    log.info(f'Time: {elapsed:.1f} min')
    log.info(f'Residuals computed: {total_residuals}')
    log.info(f'Skipped: {total_skipped}')
    log.info('='*70)

In [ ]:
# Cell 5 — Verify: check residuals on HF
from collections import defaultdict

print('Re-listing HF residuals...')
all_hf_files = list(list_repo_files(repo_id=HF_REPO, repo_type="dataset", token=HF_TOKEN))

residual_files_now = [f for f in all_hf_files if f.startswith("results/residuals/") and f.endswith(".npz")]
residual_tran_ids_now = set(f.split("/")[3].replace(".npz", "") for f in residual_files_now)

# Find which segments still need residuals
still_need = {tid for tid in expected_segments
              if tid in complete_segments and tid not in residual_tran_ids_now}

print(f'\n=== Post-06c Status ===')
print(f'Expected segments: {len(expected_segments)}')
print(f'Complete segments on HF: {len(complete_segments & set(expected_segments.keys()))}')
print(f'Residuals on HF: {len(residual_tran_ids_now)}')
print(f'Still missing residuals: {len(still_need)}')

if still_need:
    print(f'\nStill need residuals for:')
    for tid in sorted(still_need)[:20]:
        print(f'  {tid}')
    if len(still_need) > 20:
        print(f'  ... and {len(still_need) - 20} more')
else:
    print('\n✓ All complete segments have residuals!')